In [1]:

import torch
import triton
import triton.language as tl



In [6]:
@triton.jit
def add_ten_kernel(
    A, B,
    M: tl.constexpr,
    N: tl.constexpr,
    stride_am: tl.constexpr,
    stride_an: tl.constexpr,
    stride_bm: tl.constexpr,
    stride_bn: tl.constexpr,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)

    # TODO 1: Calculate global row indices
    rows = pid_m*BLOCK_M + tl.arange(0,BLOCK_M)

    # TODO 2: Calculate global column indices
    cols = pid_n*BLOCK_N + tl.arange(0,BLOCK_N)

    # TODO 3: Construct the 2D input pointers
    a_ptrs = A + (rows[:,None]*stride_am+cols[None,:]*stride_an)

    # TODO 4: Construct the 2D output pointers
    b_ptrs = B + (rows[:,None]*stride_bm+cols[None,:]*stride_bn)

    # TODO 5: Construct the 2D mask
    mask = ((rows[:,None]<M) & (cols[None,:]<N))

    # Load matrix tile
    values = tl.load(a_ptrs, mask=mask, other=0)

    # Add 10
    result = values + 10

    # TODO 6: Store the result
    tl.store(b_ptrs,result,mask=mask)

    


In [7]:

def check(A):
    M, N = A.shape
    B = torch.empty((M, N), device=A.device, dtype=A.dtype)

    grid = (
        triton.cdiv(M, 4),
        triton.cdiv(N, 4),
    )

    add_ten_kernel[grid](
        A, B,
        M, N,
        A.stride(0), A.stride(1),
        B.stride(0), B.stride(1),
        4, 4,
    )

    torch.testing.assert_close(B, A + 10)
    print("PASSED")
    print(B)


# Test 1: Contiguous matrix
A = torch.arange(
    35, device="cuda", dtype=torch.float32
).reshape(5, 7)

check(A)


PASSED
tensor([[10., 11., 12., 13., 14., 15., 16.],
        [17., 18., 19., 20., 21., 22., 23.],
        [24., 25., 26., 27., 28., 29., 30.],
        [31., 32., 33., 34., 35., 36., 37.],
        [38., 39., 40., 41., 42., 43., 44.]], device='cuda:0')


In [14]:

@triton.jit
def transpose_kernel(
    A, B,
    M: tl.constexpr,
    N: tl.constexpr,
    stride_am: tl.constexpr,
    stride_an: tl.constexpr,
    stride_bm: tl.constexpr,
    stride_bn: tl.constexpr,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)

    rows = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    cols = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)

    # Task 1: Input tile pointers
    a_ptrs = A + (rows[:,None]*stride_am + cols[None,:]*stride_an)

    # Task 2: Transposed output pointers
    b_ptrs = B + (cols[None,:]*stride_bm + rows[:,None]*stride_bn)

    # Task 3: Boundary mask
    mask = ((rows[:,None]<M) & (cols[None,:]<N))
   

    # Task 4: Load
    values = tl.load(a_ptrs,mask=mask,other=0.0)

    # Task 5: Store transposed data
    tl.store(b_ptrs,values,mask=mask)


In [15]:
A = torch.arange(
    12, device="cuda", dtype=torch.float32
).reshape(3, 4)

B = torch.empty(
    (4, 3), device="cuda", dtype=A.dtype
)

grid = (triton.cdiv(3, 4), triton.cdiv(4, 4))

transpose_kernel[grid](
    A, B,
    3, 4,
    *A.stride(),
    *B.stride(),
    4, 4,
)

torch.testing.assert_close(B, A.T)
print("PASSED")

PASSED
